
# Final Road Damage Enhancement Pipeline

## Imports

In [1]:

import cv2, os
import numpy as np
import matplotlib.pyplot as plt
import csv


## Paths

In [2]:

INPUT_DIR = "../data/frames/selected"
REPORT_DIR = "../results/report_evidence/final_pipeline"

os.makedirs(REPORT_DIR, exist_ok=True)


## Load Frames

In [3]:

images = []
names = []

for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f.lower().endswith((".jpg",".png",".jpeg")):
            p = os.path.join(root,f)
            img = cv2.imread(p,0)
            if img is not None:
                images.append(img)
                names.append(f)

print("Loaded frames:", len(images))


Loaded frames: 120


## Enhancement Functions

In [4]:

def gaussian_filter(img,k):
    return cv2.GaussianBlur(img,(k,k),0)

def clahe(img,clip):
    c = cv2.createCLAHE(clipLimit=clip,tileGridSize=(8,8))
    return c.apply(img)

def unsharp(img,k,a):
    blur = cv2.GaussianBlur(img,(k,k),0)
    return cv2.addWeighted(img,1+a,blur,-a,0)


## Final Pipeline

In [5]:

def final_pipeline(img):

    img = gaussian_filter(img,3)
    img = clahe(img,2.0)
    img = unsharp(img,7,1.0)

    return img


## Run Pipeline

In [6]:

results = []

for img in images:
    out = final_pipeline(img)
    results.append(out)


## Save Metrics CSV

In [7]:

csv_path = os.path.join(REPORT_DIR,"final_pipeline_metrics.csv")

with open(csv_path,"w",newline="") as f:

    writer = csv.writer(f)
    writer.writerow(["frame","lap_before","lap_after"])

    for img,out,nm in zip(images,results,names):

        lap_before = cv2.Laplacian(img,cv2.CV_64F).var()
        lap_after = cv2.Laplacian(out,cv2.CV_64F).var()

        writer.writerow([nm,lap_before,lap_after])

print("Report saved:", csv_path)


Report saved: ../results/report_evidence/final_pipeline\final_pipeline_metrics.csv
